# 01 — Data Validation and Market Data

This notebook is the first step of the GMAT3 relative valuation pipeline. Its objective is to validate the raw financial inputs for Grupo Mateus and its Brazilian listed peers, collect current market data, and create a clean processed dataset for later trading comparable analysis.

The workflow is intentionally simple and reproducible: first, we inspect the raw financial statement inputs; second, we collect market pricing data; third, we standardize units in BRL millions; and finally, we calculate the first valuation building blocks needed for EV/EBITDA analysis.

At this stage, the notebook does not estimate a target price or produce a final valuation conclusion. It prepares the data foundation for the next notebook in the equity research pipeline.


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
from datetime import datetime


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


The financial data comes from company earnings releases and ITR filings, then is standardized in BRL millions so the companies can be compared on the same scale.

`revenue` representa as vendas geradas pela operação antes das principais despesas. `ebitda` aproxima a geração operacional recorrente antes de depreciação, amortização, juros e impostos. `ebit` é o lucro operacional após depreciação e amortização. `net_income` é o lucro líquido após juros, impostos e demais itens abaixo da linha operacional.

`cash` is the cash and equivalents available on the balance sheet. `total_debt` is the company's gross financial debt. `shares_outstanding` is the number of shares used to connect market pricing with equity value when needed.


In [3]:
raw_data_path = Path("../data/raw/peer_financials.csv")
financials_df = pd.read_csv(raw_data_path)

financials_df


,ticker,company,sector,subsector,period,report_date,currency,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,net_revenue_growth_yoy_pct,net_income_growth_yoy_pct,same_store_sales_growth_pct,gross_margin_pct,ebitda_margin_pre_ifrs16_pct,ebitda_margin_post_ifrs16_pct,ebitda_pre_ifrs16_mn,ebitda_post_ifrs16_mn,operational_comment
0,GMAT3.SA,Grupo Mateus,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,9402,400,NaN,213,1984,2720,"2,300,047,621.00",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00,"1T26 impacted by food deflation, weaker demand..."
1,ASAI3.SA,Assai,Consumer Staples,Cash & Carry,1T26,2026-03-31,BRL,20600,1000,NaN,174,4366,16374,"1,353,531,000.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PCAR3.SA,GPA,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,4374,458,NaN,-1347,943,4173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
financials_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ticker                         3 non-null      str    
 1   company                        3 non-null      str    
 2   sector                         3 non-null      str    
 3   subsector                      3 non-null      str    
 4   period                         3 non-null      str    
 5   report_date                    3 non-null      str    
 6   currency                       3 non-null      str    
 7   revenue                        3 non-null      int64  
 8   ebitda                         3 non-null      int64  
 9   ebit                           0 non-null      float64
 10  net_income                     3 non-null      int64  
 11  cash                           3 non-null      int64  
 12  total_debt                     3 non-null      int64  
 13  share

Data type validation matters because finance models are calculation engines. If a column that should be numeric is accidentally read as text, valuation formulas may fail or silently produce incorrect results.

For trading comparable analysis, inputs such as revenue, EBITDA, debt, cash, and market capitalization must be numeric because they feed directly into margins, enterprise value, and valuation multiples.


In [5]:
financials_df.isnull().sum()


ticker                           0
company                          0
sector                           0
subsector                        0
period                           0
report_date                      0
currency                         0
revenue                          0
ebitda                           0
ebit                             3
net_income                       0
cash                             0
total_debt                       0
shares_outstanding               1
net_revenue_growth_yoy_pct       2
net_income_growth_yoy_pct        2
same_store_sales_growth_pct      2
gross_margin_pct                 2
ebitda_margin_pre_ifrs16_pct     2
ebitda_margin_post_ifrs16_pct    2
ebitda_pre_ifrs16_mn             2
ebitda_post_ifrs16_mn            2
operational_comment              2
dtype: int64

Missing values act as a financial data checklist. They show which fields still need source confirmation before the model becomes more complete.

EBIT may remain temporarily blank because this first version focuses on EV/EBITDA, the most relevant initial multiple for food retail and cash-and-carry peers. EBIT can be added later for EV/EBIT or operating margin analysis once the source data is consistently collected across all companies.


In [6]:
financials_df.describe()


,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,net_revenue_growth_yoy_pct,net_income_growth_yoy_pct,same_store_sales_growth_pct,gross_margin_pct,ebitda_margin_pre_ifrs16_pct,ebitda_margin_post_ifrs16_pct,ebitda_pre_ifrs16_mn,ebitda_post_ifrs16_mn
count,3.00,3.00,0.00,3.00,3.00,3.00,2.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
mean,"11,458.67",619.33,NaN,-320.00,"2,431.00","7,755.67","1,826,789,310.50",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00
std,"8,306.21",330.94,NaN,889.62,"1,754.73","7,498.97","669,288,321.21",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,"4,374.00",400.00,NaN,"-1,347.00",943.00,"2,720.00","1,353,531,000.00",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00
25%,"6,888.00",429.00,NaN,-586.50,"1,463.50","3,446.50","1,590,160,155.25",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00
50%,"9,402.00",458.00,NaN,174.00,"1,984.00","4,173.00","1,826,789,310.50",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00
75%,"15,001.00",729.00,NaN,193.50,"3,175.00","10,273.50","2,063,418,465.75",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00
max,"20,600.00","1,000.00",NaN,213.00,"4,366.00","16,374.00","2,300,047,621.00",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00


This sanity check helps detect scale problems before valuation work begins. A common modeling error is mixing values in absolute BRL with values in BRL millions.

Here, financial statement items such as revenue, EBITDA, cash, and debt should be in BRL millions. Very large values in these columns may indicate that a number was entered in absolute BRL instead of millions.


# Market Data

Financial statements describe operating performance and balance sheet position, but relative valuation also needs market pricing. The market price tells us how investors are valuing the company's equity today.

`stock price` is the current trading price per share. `market capitalization` is the total equity value implied by the stock market, usually calculated as share price multiplied by shares outstanding.

The valuation date matters because market prices change every trading day. A reproducible valuation model should clearly record the date when market data was collected.


In [7]:
valuation_date = datetime.today().date()
print(f"Valuation date: {valuation_date}")


Valuation date: 2026-05-31


In [8]:
market_data_records = []

for ticker in financials_df["ticker"]:
    current_price = np.nan
    market_cap = np.nan

    try:
        ticker_object = yf.Ticker(ticker)
        fast_info = ticker_object.fast_info

        current_price = fast_info.get("last_price", np.nan)
        market_cap = fast_info.get("market_cap", np.nan)

        if pd.isna(current_price) or pd.isna(market_cap):
            ticker_info = ticker_object.info
            current_price = ticker_info.get("currentPrice", current_price)
            market_cap = ticker_info.get("marketCap", market_cap)
    except Exception as error:
        print(f"Could not retrieve market data for {ticker}: {error}")

    market_data_records.append(
        {
            "ticker": ticker,
            "valuation_date": valuation_date,
            "current_price": current_price,
            "market_cap": market_cap,
        }
    )

market_df = pd.DataFrame(market_data_records)
market_df


,ticker,valuation_date,current_price,market_cap
0,GMAT3.SA,2026-05-31,4.27,9821203456
1,ASAI3.SA,2026-05-31,8.75,11740311552
2,PCAR3.SA,2026-05-31,1.86,915002432


Yahoo Finance returns market capitalization in absolute BRL for Brazilian tickers. The financial statement inputs in this notebook are standardized in BRL millions.

Before calculating enterprise value, market capitalization must be converted to BRL millions so all valuation components use the same unit.


In [9]:
master_df = financials_df.merge(
    market_df,
    on="ticker",
    how="left",
)

master_df["market_cap_mn"] = master_df["market_cap"] / 1_000_000

master_df


,ticker,company,sector,subsector,period,report_date,currency,revenue,ebitda,ebit,net_income,cash,total_debt,shares_outstanding,net_revenue_growth_yoy_pct,net_income_growth_yoy_pct,same_store_sales_growth_pct,gross_margin_pct,ebitda_margin_pre_ifrs16_pct,ebitda_margin_post_ifrs16_pct,ebitda_pre_ifrs16_mn,ebitda_post_ifrs16_mn,operational_comment,valuation_date,current_price,market_cap,market_cap_mn
0,GMAT3.SA,Grupo Mateus,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,9402,400,NaN,213,1984,2720,"2,300,047,621.00",12.90,-21.80,-7.30,22.90,4.30,5.80,399.80,543.00,"1T26 impacted by food deflation, weaker demand...",2026-05-31,4.27,9821203456,"9,821.20"
1,ASAI3.SA,Assai,Consumer Staples,Cash & Carry,1T26,2026-03-31,BRL,20600,1000,NaN,174,4366,16374,"1,353,531,000.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-31,8.75,11740311552,"11,740.31"
2,PCAR3.SA,GPA,Consumer Staples,Food Retail,1T26,2026-03-31,BRL,4374,458,NaN,-1347,943,4173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-31,1.86,915002432,915.00


# Net Debt and Enterprise Value

Market Cap is the equity value of the company: what the stock market says the shares are worth.

Net Debt is total debt minus cash. It estimates how much debt remains after using available cash to reduce financial obligations.

Enterprise Value is Market Cap plus Net Debt. Intuitively, it measures the value of the whole operating business, including both the equity owned by shareholders and the debt funded by creditors.

EV is usually superior to Market Cap for peer comparison because companies can have very different capital structures. Two retailers with similar operations may look very different on Market Cap alone if one is much more leveraged than the other.


In [10]:
master_df["net_debt"] = master_df["total_debt"] - master_df["cash"]
master_df["enterprise_value"] = master_df["market_cap_mn"] + master_df["net_debt"]

master_df[
    [
        "ticker",
        "company",
        "cash",
        "total_debt",
        "net_debt",
        "market_cap_mn",
        "enterprise_value",
    ]
]


,ticker,company,cash,total_debt,net_debt,market_cap_mn,enterprise_value
0,GMAT3.SA,Grupo Mateus,1984,2720,736,"9,821.20","10,557.20"
1,ASAI3.SA,Assai,4366,16374,12008,"11,740.31","23,748.31"
2,PCAR3.SA,GPA,943,4173,3230,915.00,"4,145.00"


# Annualized EBITDA

The current financial data is quarterly because it refers to 1T26. To create an initial comparable multiple, EBITDA is annualized as a temporary approximation.

The simple method is quarter × 4. This assumes the first quarter is representative of a full year, which may not always be true.

A future improvement is to replace this approximation with LTM EBITDA, which uses the last twelve months and generally provides a more reliable basis for valuation.


In [11]:
master_df["ebitda_annualized"] = master_df["ebitda"] * 4

master_df[
    [
        "ticker",
        "company",
        "period",
        "ebitda",
        "ebitda_annualized",
    ]
]


,ticker,company,period,ebitda,ebitda_annualized
0,GMAT3.SA,Grupo Mateus,1T26,400,1600
1,ASAI3.SA,Assai,1T26,1000,4000
2,PCAR3.SA,GPA,1T26,458,1832


# Initial Multiple Calculation

EV/EBITDA compares the total value of the business with its operating cash generation capacity.

In simple terms, it asks how much the market is paying for each unit of operating capacity. A higher multiple can reflect stronger growth, lower risk, better returns, or market optimism. A lower multiple can reflect weaker growth, higher leverage, execution concerns, or a distressed situation.


In [12]:
master_df["ev_ebitda_annualized"] = (
    master_df["enterprise_value"] / master_df["ebitda_annualized"]
)

master_df[
    [
        "ticker",
        "company",
        "revenue",
        "ebitda_annualized",
        "market_cap_mn",
        "net_debt",
        "enterprise_value",
        "ev_ebitda_annualized",
    ]
]


,ticker,company,revenue,ebitda_annualized,market_cap_mn,net_debt,enterprise_value,ev_ebitda_annualized
0,GMAT3.SA,Grupo Mateus,9402,1600,"9,821.20",736,"10,557.20",6.60
1,ASAI3.SA,Assai,20600,4000,"11,740.31",12008,"23,748.31",5.94
2,PCAR3.SA,GPA,4374,1832,915.00,3230,"4,145.00",2.26


In [13]:
display_columns = [
    "ticker",
    "company",
    "current_price",
    "revenue",
    "ebitda_annualized",
    "market_cap_mn",
    "net_debt",
    "enterprise_value",
    "ev_ebitda_annualized",
]

display_df = master_df[display_columns].copy()

monetary_columns = [
    "revenue",
    "ebitda_annualized",
    "market_cap_mn",
    "net_debt",
    "enterprise_value",
]

for column in monetary_columns:
    display_df[column] = display_df[column].apply(
        lambda value: "-" if pd.isna(value) else f"R$ {value:,.0f} mi"
    )

display_df["current_price"] = display_df["current_price"].apply(
    lambda value: "-" if pd.isna(value) else f"R$ {value:,.2f}"
)

display_df["ev_ebitda_annualized"] = display_df["ev_ebitda_annualized"].apply(
    lambda value: "-" if pd.isna(value) else f"{value:,.1f}x"
)

display_df


,ticker,company,current_price,revenue,ebitda_annualized,market_cap_mn,net_debt,enterprise_value,ev_ebitda_annualized
0,GMAT3.SA,Grupo Mateus,R$ 4.27,"R$ 9,402 mi","R$ 1,600 mi","R$ 9,821 mi",R$ 736 mi,"R$ 10,557 mi",6.6x
1,ASAI3.SA,Assai,R$ 8.75,"R$ 20,600 mi","R$ 4,000 mi","R$ 11,740 mi","R$ 12,008 mi","R$ 23,748 mi",5.9x
2,PCAR3.SA,GPA,R$ 1.86,"R$ 4,374 mi","R$ 1,832 mi",R$ 915 mi,"R$ 3,230 mi","R$ 4,145 mi",2.3x


# Initial Interpretation

Assai's enterprise value is expected to be meaningfully higher than its market capitalization because the company carries a substantial debt position. In EV-based analysis, that leverage is included in the total value of the operating business, which makes the comparison more complete than looking at equity value alone.

Grupo Mateus has a lower leverage profile in this initial dataset. That matters because a less levered balance sheet usually reduces financial risk and can make the gap between market capitalization and enterprise value smaller.

GPA should be interpreted carefully. The company presents a distressed or turnaround profile, with negative net income and a more complex operating context. Its multiple may not be directly comparable without qualitative adjustments and further normalization.

These are only first comparable signals. They do not represent a final valuation conclusion, target price, or investment recommendation. The next step is to refine the peer set, normalize earnings, and evaluate whether each multiple is economically meaningful.


In [14]:
processed_data_dir = Path("../data/processed")
processed_data_dir.mkdir(parents=True, exist_ok=True)

processed_data_path = processed_data_dir / "master_valuation_dataset.csv"
master_df.to_csv(processed_data_path, index=False)

print(f"Processed dataset exported to: {processed_data_path}")


Processed dataset exported to: ../data/processed/master_valuation_dataset.csv
